# Mirror Descent

**S&DS 432/632 -- Advanced Optimization Techniques**

This notebook accompanies Chapter 8 (Mirror Descent) of the lecture notes. It contains:

1. **Bregman divergence visualization** -- comparing Euclidean and KL geometries
2. **Mirror descent on the simplex** -- exponentiated gradient vs. projected gradient descent
3. **Dimension-free rates** -- empirical demonstration of $O(\sqrt{\log n / T})$ vs. $O(\sqrt{n / T})$
4. **Online learning / experts problem** -- multiplicative weights and cumulative regret
5. **Comparison of mirror maps** -- convergence curves for different geometries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.optimize import linprog

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Morandi color palette
TERRACOTTA = '#C47A6A'
SAGE       = '#7A9A7E'
STEEL_BLUE = '#7B97AD'
GOLD       = '#B8995E'
LAVENDER   = '#907EA0'

## 1. Bregman Divergence Visualization

The **Bregman divergence** induced by a strictly convex, differentiable function $\varphi$ is defined as

$$D_\varphi(x, y) = \varphi(x) - \varphi(y) - \langle \nabla \varphi(y),\, x - y \rangle.$$

Geometrically, $D_\varphi(x, y)$ is the vertical gap between the graph of $\varphi$ at $x$ and its tangent approximation anchored at $y$. By convexity, this gap is always nonnegative, and it equals zero only when $x = y$.

Two important special cases:

| Mirror map $\varphi$ | Bregman divergence $D_\varphi(x,y)$ |
|---|---|
| $\varphi(x) = \frac{1}{2}\|x\|_2^2$ (Euclidean) | $\frac{1}{2}\|x - y\|_2^2$ |
| $\varphi(x) = \sum_i x_i \log x_i$ (neg. entropy) | $\mathrm{KL}(x, y) = \sum_i x_i \log(x_i / y_i)$ |

We visualize both in 1D below.

In [ ]:
# --- Bregman divergence visualization ---

def bregman_plot(phi, dphi, x_range, y_anchor, title, ax):
    """Plot phi(x), its tangent at y_anchor, and shade the Bregman divergence."""
    x = np.linspace(*x_range, 500)
    phi_vals = phi(x)
    tangent = phi(y_anchor) + dphi(y_anchor) * (x - y_anchor)

    ax.plot(x, phi_vals, color='k', linewidth=2.5, label=r'$\varphi(x)$')
    ax.plot(x, tangent, '--', color=STEEL_BLUE, linewidth=2,
            label=f'Tangent at $y = {y_anchor}$')

    # Shade the Bregman divergence region between phi and tangent
    mask = phi_vals >= tangent
    ax.fill_between(x, tangent, phi_vals, where=mask,
                    alpha=0.25, color=TERRACOTTA, label=r'$D_\varphi(x, y)$')

    # Mark the anchor point
    ax.plot(y_anchor, phi(y_anchor), 'o', color=SAGE, markersize=10, zorder=5)

    # Mark an example point and draw the vertical gap
    x_ex = y_anchor + 0.6 * (x_range[1] - y_anchor)
    ax.annotate('', xy=(x_ex, phi(x_ex)),
                xytext=(x_ex, phi(y_anchor) + dphi(y_anchor) * (x_ex - y_anchor)),
                arrowprops=dict(arrowstyle='<->', color=TERRACOTTA, lw=2))
    ax.text(x_ex + 0.08,
            0.5 * (phi(x_ex) + phi(y_anchor) + dphi(y_anchor) * (x_ex - y_anchor)),
            r'$D_\varphi$', fontsize=14, color=TERRACOTTA, fontweight='bold')

    ax.set_xlabel('$x$')
    ax.set_ylabel(r'$\varphi(x)$')
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# (a) Euclidean: phi(x) = x^2 / 2
bregman_plot(
    phi=lambda x: 0.5 * x**2,
    dphi=lambda x: x,
    x_range=(-2, 3),
    y_anchor=0.5,
    title=r'Euclidean: $\varphi(x) = \frac{1}{2}x^2$',
    ax=ax1
)

# (b) Negative entropy: phi(x) = x log(x) for x > 0
bregman_plot(
    phi=lambda x: x * np.log(x),
    dphi=lambda x: np.log(x) + 1,
    x_range=(0.05, 3),
    y_anchor=0.8,
    title=r'Neg. entropy: $\varphi(x) = x \log x$',
    ax=ax2
)

plt.tight_layout()
plt.show()

### Asymmetry of Bregman Divergence

Unlike the Euclidean distance, the Bregman divergence is **not symmetric**: $D_\varphi(x, y) \neq D_\varphi(y, x)$ in general. We demonstrate this for the negative entropy.

In [ ]:
# --- Asymmetry of KL divergence ---

def kl_1d(x, y):
    """KL divergence for two scalars on (0, inf) using phi(t) = t log t."""
    return x * np.log(x / y) - x + y

x_grid = np.linspace(0.1, 3.0, 300)
y0 = 1.0

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_grid, kl_1d(x_grid, y0), linewidth=2.5, color=TERRACOTTA,
        label=r'$D_\varphi(x,\, y{=}1)$')
ax.plot(x_grid, kl_1d(y0, x_grid), linewidth=2.5, color=SAGE,
        label=r'$D_\varphi(y{=}1,\, x)$')
ax.plot(x_grid, 0.5 * (x_grid - y0)**2, '--', linewidth=2, color=STEEL_BLUE,
        label=r'$\frac{1}{2}(x - y)^2$ (Euclidean, symmetric)')
ax.set_xlabel('$x$')
ax.set_ylabel('Divergence')
ax.set_title('Asymmetry of the KL Bregman Divergence ($y = 1$)')
ax.legend()
ax.set_ylim(0, 4)
plt.tight_layout()
plt.show()

**Interpretation.** The terracotta and sage curves are the two "directions" of the KL divergence. They differ markedly: $D_\varphi(x, 1)$ penalizes $x \gg 1$ more heavily (like an exponential tail), while $D_\varphi(1, x)$ penalizes $x \to 0^+$ more severely. In contrast, the Euclidean distance (dashed) is perfectly symmetric.

## 2. Mirror Descent on the Simplex

We now implement mirror descent for minimizing a linear function $f(x) = c^\top x$ over the probability simplex

$$\Delta_n = \Big\{ x \in \mathbb{R}^n : x \geq 0,\; \sum_{i=1}^n x_i = 1 \Big\}.$$

### Mirror descent with negative entropy (exponentiated gradient)

Using the mirror map $\varphi(x) = \sum_i x_i \log x_i$, the mirror descent update becomes the **multiplicative weights** (exponentiated gradient) rule:

$$x_i^{k+1} = \frac{x_i^k \cdot \exp\!\big(-\alpha \cdot \nabla_i f(x^k)\big)}{\sum_{j=1}^n x_j^k \cdot \exp\!\big(-\alpha \cdot \nabla_j f(x^k)\big)}.$$

### Projected gradient descent (Euclidean projection)

Using $\varphi(x) = \frac{1}{2}\|x\|_2^2$, the update is gradient descent followed by Euclidean projection onto $\Delta_n$:

$$x^{k+1} = \Pi_{\Delta_n}\!\big(x^k - \alpha \nabla f(x^k)\big).$$

The Euclidean projection onto the simplex can be computed in $O(n \log n)$ time by sorting.

In [ ]:
# --- Core algorithms ---

def project_simplex(v):
    """Euclidean projection of v onto the probability simplex.

    Algorithm: sort in descending order, then find the threshold
    (Duchi et al., 2008).
    """
    n = len(v)
    u = np.sort(v)[::-1]
    cumsum = np.cumsum(u)
    rho = np.where(u > (cumsum - 1) / np.arange(1, n + 1))[0][-1]
    theta = (cumsum[rho] - 1) / (rho + 1)
    return np.maximum(v - theta, 0)


def mirror_descent_entropy(grad_f, x0, alpha, n_iter):
    """Mirror descent with negative-entropy mirror map (exponentiated gradient).

    Parameters
    ----------
    grad_f : callable, returns gradient at x
    x0     : initial point on the simplex
    alpha  : step size (constant)
    n_iter : number of iterations

    Returns
    -------
    iterates : list of arrays, length n_iter + 1
    """
    x = x0.copy()
    iterates = [x.copy()]
    for _ in range(n_iter):
        g = grad_f(x)
        # Multiplicative weights update
        x = x * np.exp(-alpha * g)
        x = x / x.sum()  # normalize to stay on simplex
        iterates.append(x.copy())
    return iterates


def projected_gd_simplex(grad_f, x0, alpha, n_iter):
    """Projected gradient descent with Euclidean projection onto the simplex.

    Parameters
    ----------
    grad_f : callable, returns gradient at x
    x0     : initial point on the simplex
    alpha  : step size (constant)
    n_iter : number of iterations

    Returns
    -------
    iterates : list of arrays, length n_iter + 1
    """
    x = x0.copy()
    iterates = [x.copy()]
    for _ in range(n_iter):
        g = grad_f(x)
        x = project_simplex(x - alpha * g)
        iterates.append(x.copy())
    return iterates

### Demo: Linear objective on the simplex

We minimize $f(x) = c^\top x$ over $\Delta_n$ where $c \in \mathbb{R}^n$ has one coordinate much smaller than the rest. The optimal solution concentrates all mass on the coordinate with the smallest $c_i$.

In [ ]:
# --- Linear objective on the simplex ---
np.random.seed(42)
n = 20
c = np.random.uniform(0.5, 1.5, size=n)
c[0] = 0.0  # make coordinate 0 the unique minimizer

grad_f = lambda x: c  # gradient of c^T x is just c
f_val = lambda x: c @ x
f_star = 0.0  # optimum is e_1, giving f = c[0] = 0

x0 = np.ones(n) / n  # uniform initialization
n_iter = 300
alpha_md = 0.5
alpha_pgd = 0.05

iters_md = mirror_descent_entropy(grad_f, x0, alpha=alpha_md, n_iter=n_iter)
iters_pgd = projected_gd_simplex(grad_f, x0, alpha=alpha_pgd, n_iter=n_iter)

# Compute suboptimality
vals_md = [f_val(x) - f_star for x in iters_md]
vals_pgd = [f_val(x) - f_star for x in iters_pgd]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: suboptimality
ax1.semilogy(vals_md, linewidth=2, color=TERRACOTTA, label='Mirror Descent (entropy)')
ax1.semilogy(vals_pgd, linewidth=2, color=STEEL_BLUE, label='Projected GD (Euclidean)')
ax1.set_xlabel('Iteration $k$')
ax1.set_ylabel('$f(x^k) - f^*$')
ax1.set_title('Suboptimality: Linear Objective on $\\Delta_{20}$')
ax1.legend()

# Right: evolution of x[0] (the optimal coordinate)
ax2.plot([x[0] for x in iters_md], linewidth=2, color=TERRACOTTA,
         label='MD: $x_1^k$')
ax2.plot([x[0] for x in iters_pgd], linewidth=2, color=STEEL_BLUE,
         label='PGD: $x_1^k$')
ax2.axhline(1.0, color='gray', linestyle=':', alpha=0.6, label='Optimal $x_1^* = 1$')
ax2.set_xlabel('Iteration $k$')
ax2.set_ylabel('$x_1^k$')
ax2.set_title('Weight on Optimal Coordinate')
ax2.legend()

plt.tight_layout()
plt.show()

**Interpretation.** Mirror descent with the entropy mirror map uses multiplicative updates, naturally keeping the iterates positive and on the simplex. It redistributes probability mass by *multiplicatively* amplifying coordinates with small gradient entries and shrinking those with large ones. Projected GD, by contrast, takes an unconstrained gradient step and then "clips" back to the simplex via Euclidean projection -- a less geometry-aware operation.

## 3. Dimension-Free Rates

A central result of the mirror descent theory is the different dimension dependence when optimizing over $\Delta_n$:

| Method | Mirror map | Rate |
|---|---|---|
| Projected subgradient | $\varphi(x) = \frac{1}{2}\|x\|_2^2$ | $O\!\big(L_{f,2} / \sqrt{T}\big)$, where $R = O(1)$ |
| Mirror descent (entropy) | $\varphi(x) = \sum x_i \log x_i$ | $O\!\big(L_{f,\infty} \sqrt{\log n / T}\big)$, where $R = \log n$ |

Since $\|g\|_\infty \leq \|g\|_2 \leq \sqrt{n}\,\|g\|_\infty$, the key question is:

> When $\|g\|_2 \approx \sqrt{n}\,\|g\|_\infty$ (i.e., the gradient is "spread out"), PGD's rate scales as $O(\sqrt{n/T})$ while MD's scales as $O(\sqrt{\log n / T})$ -- an **exponential** improvement in dimension.

We verify this empirically by running both methods for increasing $n$ and measuring the final suboptimality.

In [ ]:
# --- Dimension dependence experiment ---
# We use a "hard" linear objective where the gradient is spread out:
#   c_i = 1 + small_noise for all i except c[0] = 0
# This makes ||c||_2 ~ sqrt(n) while ||c||_inf ~ 1.

dims = [10, 20, 50, 100, 200, 500, 1000]
T = 500  # fixed number of iterations

final_gap_md = []
final_gap_pgd = []

for n in dims:
    # Construct cost vector: c_i ~ 1 for i >= 1, c_0 = 0
    rng = np.random.default_rng(0)
    c = np.ones(n) + 0.1 * rng.standard_normal(n)
    c = np.abs(c)  # ensure positivity
    c[0] = 0.0
    f_star = 0.0

    grad_f = lambda x, c=c: c
    f_val = lambda x, c=c: c @ x
    x0 = np.ones(n) / n

    # Step sizes from theory:
    #   MD:  alpha ~ sqrt(log(n) / T)
    #   PGD: alpha ~ 1 / (||c||_2 * sqrt(T))
    alpha_md = np.sqrt(np.log(n) / T)
    alpha_pgd = 1.0 / (np.linalg.norm(c) * np.sqrt(T))

    iters_md = mirror_descent_entropy(grad_f, x0, alpha=alpha_md, n_iter=T)
    iters_pgd = projected_gd_simplex(grad_f, x0, alpha=alpha_pgd, n_iter=T)

    # Use the average iterate (as suggested by theory for the Lipschitz case)
    x_avg_md = np.mean(iters_md[1:], axis=0)
    x_avg_pgd = np.mean(iters_pgd[1:], axis=0)

    final_gap_md.append(f_val(x_avg_md) - f_star)
    final_gap_pgd.append(f_val(x_avg_pgd) - f_star)

# Theoretical reference curves
dims_arr = np.array(dims, dtype=float)
ref_logn = 0.8 * np.sqrt(np.log(dims_arr) / T)   # O(sqrt(log n / T))
ref_sqrtn = 0.5 * np.sqrt(dims_arr / T)           # O(sqrt(n / T))

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(dims, final_gap_md, 'o-', color=TERRACOTTA, linewidth=2, markersize=8,
          label='Mirror Descent (entropy)')
ax.loglog(dims, final_gap_pgd, 's-', color=STEEL_BLUE, linewidth=2, markersize=8,
          label='Projected GD (Euclidean)')
ax.loglog(dims, ref_logn, '--', color=TERRACOTTA, alpha=0.5,
          label=r'$O(\sqrt{\log n \,/\, T})$')
ax.loglog(dims, ref_sqrtn, '--', color=STEEL_BLUE, alpha=0.5,
          label=r'$O(\sqrt{n \,/\, T})$')

ax.set_xlabel('Dimension $n$')
ax.set_ylabel('Suboptimality of average iterate')
ax.set_title(f'Dimension Dependence ($T = {T}$ iterations)')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**Interpretation.** The log-log plot confirms the theoretical prediction. As $n$ grows:

- **Projected GD** (blue squares) degrades rapidly, tracking the $O(\sqrt{n/T})$ reference line. The Euclidean distance $R = \sup_{x \in \Delta_n} \frac{1}{2}\|x - \mathbf{1}/n\|_2^2 \leq 1/2$ is dimension-free, but the Lipschitz constant $\|c\|_2 \sim \sqrt{n}$ absorbs the dimension.
- **Mirror descent** (terracotta circles) barely increases, tracking $O(\sqrt{\log n / T})$. The KL diameter $R = \sup_{x \in \Delta_n} \mathrm{KL}(x, \mathbf{1}/n) \leq \log n$ grows slowly, and the Lipschitz constant $\|c\|_\infty \sim 1$ is dimension-free.

This exponential gap ($\sqrt{n}$ vs. $\sqrt{\log n}$) is the central motivation for mirror descent.

## 4. Online Learning / Experts Problem

In the **prediction with expert advice** (experts) problem, a learner repeatedly chooses a distribution over $n$ experts. At each round $t$:

1. The learner plays $x^t \in \Delta_n$.
2. An adversary reveals a loss vector $\ell^t \in [0, 1]^n$.
3. The learner suffers expected loss $\langle \ell^t, x^t \rangle$.

The **regret** measures the gap between the learner's total loss and the best single expert in hindsight:

$$\mathrm{Regret}_T = \sum_{t=1}^{T} \langle \ell^t, x^t \rangle - \min_{i \in [n]} \sum_{t=1}^{T} \ell^t_i.$$

**Online mirror descent** with the entropy mirror map yields the celebrated **multiplicative weights algorithm** (also known as Hedge / exponentiated gradient):

$$x_i^{t+1} = \frac{x_i^t \cdot \exp(-\eta \,\ell_i^t)}{\sum_j x_j^t \cdot \exp(-\eta\, \ell_j^t)}.$$

The regret bound is:

$$\mathrm{Regret}_T \leq \frac{\log n}{\eta} + \frac{\eta T}{2} = O\!\big(\sqrt{T \log n}\big) \quad \text{with } \eta = \sqrt{\frac{2\log n}{T}}.$$

In [ ]:
# --- Online learning: experts problem ---

def multiplicative_weights(losses, eta):
    """Run the multiplicative weights (Hedge) algorithm.

    Parameters
    ----------
    losses : ndarray of shape (T, n), adversarial loss vectors in [0, 1]
    eta    : learning rate

    Returns
    -------
    weights      : list of weight vectors (T+1 entries)
    cum_regret   : cumulative regret at each round (length T)
    """
    T, n = losses.shape
    x = np.ones(n) / n
    weights = [x.copy()]
    cum_loss_learner = 0.0
    cum_loss_experts = np.zeros(n)
    cum_regret = []

    for t in range(T):
        # Learner's loss
        cum_loss_learner += x @ losses[t]
        cum_loss_experts += losses[t]

        # Regret = learner's total - best expert's total
        cum_regret.append(cum_loss_learner - cum_loss_experts.min())

        # Multiplicative weights update
        x = x * np.exp(-eta * losses[t])
        x = x / x.sum()
        weights.append(x.copy())

    return weights, np.array(cum_regret)


def online_pgd(losses, eta):
    """Run online projected gradient descent (Euclidean).

    Parameters
    ----------
    losses : ndarray of shape (T, n)
    eta    : learning rate

    Returns
    -------
    weights      : list of weight vectors
    cum_regret   : cumulative regret at each round
    """
    T, n = losses.shape
    x = np.ones(n) / n
    weights = [x.copy()]
    cum_loss_learner = 0.0
    cum_loss_experts = np.zeros(n)
    cum_regret = []

    for t in range(T):
        cum_loss_learner += x @ losses[t]
        cum_loss_experts += losses[t]
        cum_regret.append(cum_loss_learner - cum_loss_experts.min())

        # PGD update
        x = project_simplex(x - eta * losses[t])
        weights.append(x.copy())

    return weights, np.array(cum_regret)

In [ ]:
# --- Adversarial loss sequence ---
# The adversary picks one "good" expert at random each round and gives it
# low loss; all others get high loss. This makes the problem hard.

np.random.seed(123)
n_experts = 50
T_online = 2000

losses = np.random.uniform(0.3, 0.7, size=(T_online, n_experts))
# Make expert 0 consistently best (but not always the best in each round)
losses[:, 0] = np.random.uniform(0.0, 0.3, size=T_online)

# Optimal step sizes
eta_mw = np.sqrt(2 * np.log(n_experts) / T_online)
eta_pgd_online = 1.0 / np.sqrt(T_online)

weights_mw, regret_mw = multiplicative_weights(losses, eta=eta_mw)
weights_pgd, regret_pgd = online_pgd(losses, eta=eta_pgd_online)

# Theoretical regret bounds
t_arr = np.arange(1, T_online + 1)
bound_mw = np.sqrt(2 * t_arr * np.log(n_experts))  # O(sqrt(T log n))
bound_pgd = np.sqrt(n_experts * t_arr)               # O(sqrt(n T))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative regret
ax1.plot(t_arr, regret_mw, color=TERRACOTTA, linewidth=2,
         label='MW (mirror descent)')
ax1.plot(t_arr, regret_pgd, color=STEEL_BLUE, linewidth=2,
         label='Online PGD')
ax1.plot(t_arr, bound_mw, '--', color=TERRACOTTA, alpha=0.5,
         label=r'$\sqrt{2T\log n}$ bound')
ax1.plot(t_arr, bound_pgd, '--', color=STEEL_BLUE, alpha=0.5,
         label=r'$\sqrt{nT}$ bound')
ax1.set_xlabel('Round $t$')
ax1.set_ylabel('Cumulative regret')
ax1.set_title(f'Cumulative Regret ($n = {n_experts}$ experts)')
ax1.legend(fontsize=10)

# Right: weight evolution for MW on the best expert
ax2.plot([w[0] for w in weights_mw[:500]], color=TERRACOTTA, linewidth=2,
         label='MW: $x_1^t$ (best expert)')
ax2.plot([w[0] for w in weights_pgd[:500]], color=STEEL_BLUE, linewidth=2,
         label='PGD: $x_1^t$ (best expert)')
ax2.set_xlabel('Round $t$')
ax2.set_ylabel('Weight on expert 1')
ax2.set_title('Weight Concentration (first 500 rounds)')
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Final regret -- MW: {regret_mw[-1]:.2f}, PGD: {regret_pgd[-1]:.2f}')
print(f'Theoretical bound -- MW: {bound_mw[-1]:.2f}, PGD: {bound_pgd[-1]:.2f}')

**Interpretation.** The left panel shows that multiplicative weights (= online mirror descent with entropy) achieves much lower cumulative regret than online PGD, and both stay below their respective theoretical bounds. The right panel shows that MW quickly concentrates weight on the best expert through multiplicative amplification, while PGD's additive updates are slower to concentrate.

The key insight: in the online setting, the regret of MW scales as $O(\sqrt{T \log n})$ regardless of the adversary's strategy, while PGD suffers $O(\sqrt{nT})$ regret -- the same exponential gap in $n$ that we saw in the offline setting.

### Regret scaling with number of experts

We now fix $T$ and vary $n$ to directly observe the $\sqrt{\log n}$ vs. $\sqrt{n}$ scaling of the final regret.

In [ ]:
# --- Regret vs. number of experts ---
T_fixed = 1000
n_values = [5, 10, 20, 50, 100, 200, 500]

regret_mw_vs_n = []
regret_pgd_vs_n = []

for n_exp in n_values:
    rng = np.random.default_rng(42)
    loss_seq = rng.uniform(0.3, 0.7, size=(T_fixed, n_exp))
    loss_seq[:, 0] = rng.uniform(0.0, 0.3, size=T_fixed)

    eta_mw = np.sqrt(2 * np.log(n_exp) / T_fixed)
    eta_pgd = 1.0 / np.sqrt(T_fixed)

    _, reg_mw = multiplicative_weights(loss_seq, eta=eta_mw)
    _, reg_pgd = online_pgd(loss_seq, eta=eta_pgd)

    regret_mw_vs_n.append(reg_mw[-1])
    regret_pgd_vs_n.append(reg_pgd[-1])

n_arr = np.array(n_values, dtype=float)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(n_values, regret_mw_vs_n, 'o-', color=TERRACOTTA, linewidth=2,
          markersize=8, label='MW (entropy)')
ax.loglog(n_values, regret_pgd_vs_n, 's-', color=STEEL_BLUE, linewidth=2,
          markersize=8, label='Online PGD')

# Reference lines
scale_mw = regret_mw_vs_n[0] / np.sqrt(np.log(n_arr[0]))
scale_pgd = regret_pgd_vs_n[0] / np.sqrt(n_arr[0])
ax.loglog(n_arr, scale_mw * np.sqrt(np.log(n_arr)), '--', color=TERRACOTTA,
          alpha=0.5, label=r'$\propto \sqrt{\log n}$')
ax.loglog(n_arr, scale_pgd * np.sqrt(n_arr), '--', color=STEEL_BLUE,
          alpha=0.5, label=r'$\propto \sqrt{n}$')

ax.set_xlabel('Number of experts $n$')
ax.set_ylabel(f'Final regret at $T = {T_fixed}$')
ax.set_title('Regret Scaling with Dimension')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 5. Comparison of Mirror Maps

We now compare three different algorithms for a constrained optimization problem on the simplex, each corresponding to a different choice of mirror map / projection:

| Label | Mirror map $\varphi$ | Update rule |
|---|---|---|
| **(a) GD + Euclidean projection** | $\frac{1}{2}\|x\|_2^2$ | $x^{k+1} = \Pi_{\Delta_n}(x^k - \alpha \nabla f(x^k))$ |
| **(b) MD with negative entropy** | $\sum x_i \log x_i$ | Exponentiated gradient (multiplicative weights) |
| **(c) MD with $\ell_2^2/2$** | $\frac{1}{2}\|x\|_2^2$ | Same as (a) -- this shows that MD with the Euclidean mirror map *is* projected GD |

We also include a variant **(d) MD with scaled entropy** to show the effect of step-size tuning.

### Test problem: quadratic objective on the simplex

We minimize $f(x) = \frac{1}{2} x^\top Q x + q^\top x$ over $\Delta_n$, which is a smooth convex function with easily computable gradients.

In [ ]:
# --- Comparison of mirror maps ---

def make_quadratic_simplex_problem(n, seed=0):
    """Create a quadratic objective on the simplex:
       f(x) = 0.5 * x^T Q x + q^T x
    where Q is positive definite, ensuring a unique optimum on Delta_n.
    """
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((n, n))
    Q = A.T @ A / n + 0.5 * np.eye(n)  # ensure well-conditioned PD
    q = rng.standard_normal(n)

    def f(x):
        return 0.5 * x @ Q @ x + q @ x

    def grad_f(x):
        return Q @ x + q

    # Compute optimum via projected gradient descent with many iterations
    x_star = np.ones(n) / n
    for _ in range(10000):
        x_star = project_simplex(x_star - 0.001 * grad_f(x_star))
    f_star = f(x_star)

    return f, grad_f, f_star, Q


n = 50
f_obj, grad_obj, f_star, Q_mat = make_quadratic_simplex_problem(n, seed=42)
x0 = np.ones(n) / n
n_iter = 500

# (a) Projected GD (= MD with l2^2 / 2 mirror map)
alpha_pgd = 0.01
iters_pgd = projected_gd_simplex(grad_obj, x0, alpha=alpha_pgd, n_iter=n_iter)
gaps_pgd = [f_obj(x) - f_star for x in iters_pgd]

# (b) MD with negative entropy, small step
alpha_md_small = 0.1
iters_md_small = mirror_descent_entropy(grad_obj, x0, alpha=alpha_md_small, n_iter=n_iter)
gaps_md_small = [f_obj(x) - f_star for x in iters_md_small]

# (c) MD with negative entropy, tuned step
alpha_md_tuned = 0.5
iters_md_tuned = mirror_descent_entropy(grad_obj, x0, alpha=alpha_md_tuned, n_iter=n_iter)
gaps_md_tuned = [f_obj(x) - f_star for x in iters_md_tuned]

# (d) MD with negative entropy, aggressive step
alpha_md_agg = 1.0
iters_md_agg = mirror_descent_entropy(grad_obj, x0, alpha=alpha_md_agg, n_iter=n_iter)
gaps_md_agg = [f_obj(x) - f_star for x in iters_md_agg]

fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(gaps_pgd, linewidth=2.5, color=STEEL_BLUE,
            label=f'PGD (Euclidean proj.), $\\alpha={alpha_pgd}$')
ax.semilogy(gaps_md_small, linewidth=2.5, color=SAGE,
            label=f'MD (entropy), $\\alpha={alpha_md_small}$')
ax.semilogy(gaps_md_tuned, linewidth=2.5, color=TERRACOTTA,
            label=f'MD (entropy), $\\alpha={alpha_md_tuned}$')
ax.semilogy(gaps_md_agg, linewidth=2.5, color=GOLD,
            label=f'MD (entropy), $\\alpha={alpha_md_agg}$')

ax.set_xlabel('Iteration $k$')
ax.set_ylabel('$f(x^k) - f^*$')
ax.set_title(f'Mirror Map Comparison: Quadratic on $\\Delta_{{{n}}}$')
ax.legend(fontsize=11)
ax.set_ylim(bottom=1e-10)
plt.tight_layout()
plt.show()

**Interpretation.** For the quadratic objective on $\Delta_n$:

- **Projected GD** (blue) converges steadily -- it is equivalent to mirror descent with the Euclidean mirror map $\varphi(x) = \frac{1}{2}\|x\|_2^2$.
- **Mirror descent with entropy** (terracotta, sage, gold) shows the effect of step-size tuning. With a well-chosen step size, the entropy mirror map can outperform PGD because it naturally exploits the simplex geometry -- the iterates remain strictly in the interior of $\Delta_n$ and never hit the boundary, which can help for problems where the optimum has many nonzero coordinates.
- The choice of step size is crucial: too small ($\alpha = 0.1$) converges slowly; too large ($\alpha = 1.0$) may oscillate initially before settling.

### Visualizing the iterate trajectories on $\Delta_3$

To build geometric intuition, we project the problem down to $n = 3$ dimensions where we can visualize the simplex directly. We plot the iterates of PGD and MD (entropy) on the 2D simplex using barycentric coordinates.

In [ ]:
# --- Simplex trajectory visualization in 2D ---

def bary_to_cart(p):
    """Convert barycentric coordinates (3-simplex) to 2D Cartesian."""
    # Vertices of equilateral triangle
    v0 = np.array([0, 0])
    v1 = np.array([1, 0])
    v2 = np.array([0.5, np.sqrt(3) / 2])
    return p[0] * v0 + p[1] * v1 + p[2] * v2

# Small problem on Delta_3
n3 = 3
f3, grad3, f_star3, _ = make_quadratic_simplex_problem(n3, seed=7)
x0_3 = np.ones(n3) / n3

iters_pgd_3 = projected_gd_simplex(grad3, x0_3, alpha=0.05, n_iter=100)
iters_md_3 = mirror_descent_entropy(grad3, x0_3, alpha=0.3, n_iter=100)

# Compute the optimum for plotting
x_star_3 = np.ones(n3) / n3
for _ in range(50000):
    x_star_3 = project_simplex(x_star_3 - 0.001 * grad3(x_star_3))

fig, ax = plt.subplots(figsize=(8, 7))

# Draw simplex boundary
tri_verts = np.array([bary_to_cart(np.eye(3)[i]) for i in range(3)])
triangle = plt.Polygon(tri_verts, fill=False, edgecolor='gray', linewidth=1.5)
ax.add_patch(triangle)

# Label vertices
offset = 0.04
ax.text(tri_verts[0, 0] - offset, tri_verts[0, 1] - offset, '$e_1$',
        fontsize=14, ha='center')
ax.text(tri_verts[1, 0] + offset, tri_verts[1, 1] - offset, '$e_2$',
        fontsize=14, ha='center')
ax.text(tri_verts[2, 0], tri_verts[2, 1] + offset, '$e_3$',
        fontsize=14, ha='center')

# Plot contours of the objective on the simplex
ngrid = 200
xx, yy = np.meshgrid(np.linspace(-0.05, 1.05, ngrid),
                     np.linspace(-0.05, 1.0, ngrid))
zz = np.full_like(xx, np.nan)
for i in range(ngrid):
    for j in range(ngrid):
        # Invert barycentric: solve for p given (x, y)
        x_pt, y_pt = xx[i, j], yy[i, j]
        p2 = y_pt / (np.sqrt(3) / 2)
        p1 = x_pt - 0.5 * p2
        p0 = 1 - p1 - p2
        if p0 >= -0.01 and p1 >= -0.01 and p2 >= -0.01:
            p = np.array([max(p0, 0), max(p1, 0), max(p2, 0)])
            p = p / p.sum()
            zz[i, j] = f3(p)

ax.contour(xx, yy, zz, levels=20, colors='gray', alpha=0.3, linewidths=0.8)

# Plot PGD iterates
pts_pgd = np.array([bary_to_cart(x) for x in iters_pgd_3[:60]])
ax.plot(pts_pgd[:, 0], pts_pgd[:, 1], 'o-', color=STEEL_BLUE,
        markersize=4, linewidth=1.5, alpha=0.8, label='PGD')

# Plot MD iterates
pts_md = np.array([bary_to_cart(x) for x in iters_md_3[:60]])
ax.plot(pts_md[:, 0], pts_md[:, 1], 's-', color=TERRACOTTA,
        markersize=4, linewidth=1.5, alpha=0.8, label='MD (entropy)')

# Mark optimum
pt_star = bary_to_cart(x_star_3)
ax.plot(pt_star[0], pt_star[1], '*', color=GOLD, markersize=18,
        markeredgecolor='k', markeredgewidth=0.5, zorder=10, label='$x^*$')

# Mark start
pt_start = bary_to_cart(x0_3)
ax.plot(pt_start[0], pt_start[1], 'D', color=SAGE, markersize=10,
        markeredgecolor='k', markeredgewidth=0.5, zorder=10, label='$x^0$')

ax.set_xlim(-0.1, 1.1)
ax.set_ylim(-0.1, 1.0)
ax.set_aspect('equal')
ax.set_title('Iterate Trajectories on $\\Delta_3$')
ax.legend(fontsize=11, loc='upper right')
ax.grid(False)
ax.axis('off')
plt.tight_layout()
plt.show()

**Interpretation.** The 2D simplex visualization reveals the geometric difference between the two methods:

- **PGD** (blue) takes gradient steps and then projects back to the simplex boundary using the shortest Euclidean distance. When iterates hit a face of the simplex, they "slide along" the boundary.
- **MD with entropy** (terracotta) follows curved paths through the simplex *interior*. The multiplicative update ensures all coordinates remain strictly positive, so the iterates never touch the boundary -- they approach it exponentially but never reach it.

Both converge to the same optimum $x^*$ (gold star), but along qualitatively different paths.